# Final model: LightGBM with optuna, window feat, kmeans (k=3)

In [1]:
import os
import re
import gc
import ctypes

import numpy as np
import pandas as pd

import lightgbm as lgb

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

In [2]:
path_to_data = os.path.expanduser("~/PycharmProjects/home-credit-default-risk/data/processed/")
train_path = os.path.join(path_to_data, "application_train.csv")
test_path = os.path.join(path_to_data, "application_test.csv")

In [3]:
def free_memory():
    gc.collect()

    try:
        ctypes.CDLL(
            "libc.so.6"
        ).malloc_trim(0)
    except OSError:
        pass


def show_ram():
    with open(
        "/proc/self/status"
    ) as f:

        for line in f:

            if line.startswith("VmRSS:"):

                ram_kb = int(
                    line.split()[1]
                )

                ram_gb = (
                    ram_kb
                    / 1024
                    / 1024
                )

                print(
                    f"RAM Python: "
                    f"{ram_gb:.2f} ГБ"
                )

                break

In [4]:
def read_csv_float32(
        file_path,
        chunksize=10000
):

    chunks = []

    for i, chunk in enumerate(
        pd.read_csv(
            file_path,
            chunksize=chunksize
        )
    ):

        float_cols = (
            chunk
            .select_dtypes(
                include=["float64"]
            )
            .columns
        )

        chunk[
            float_cols
        ] = (
            chunk[
                float_cols
            ]
            .astype("float32")
        )

        chunks.append(
            chunk
        )

        if (i + 1) % 10 == 0:
            print(
                f"Прочитано частей: "
                f"{i + 1}"
            )

    df = pd.concat(
        chunks,
        axis=0,
        ignore_index=True,
        copy=False
    )

    del chunks

    free_memory()

    return df

In [5]:
%%time
application_train = (read_csv_float32(train_path))
print("Train:", application_train.shape)
show_ram()

Прочитано частей: 10
Прочитано частей: 20
Прочитано частей: 30
Train: (307511, 1164)
RAM Python: 1.70 ГБ
CPU times: user 18 s, sys: 3.88 s, total: 21.9 s
Wall time: 24.1 s


In [6]:
%%time
application_test = (read_csv_float32(test_path))
print("Test:", application_test.shape)
show_ram()

Test: (48744, 1163)
RAM Python: 1.92 ГБ
CPU times: user 3.17 s, sys: 419 ms, total: 3.59 s
Wall time: 3.97 s


In [7]:
y_full = (application_train["TARGET"].copy())
train_ids = (application_train["SK_ID_CURR"].copy())
test_ids = (application_test["SK_ID_CURR"].copy())

In [8]:
X_full = (application_train.drop(columns=["TARGET", "SK_ID_CURR"]))
X_test = (application_test.drop(columns=["SK_ID_CURR"]))

del application_train
del application_test

free_memory()

print("X_full:",X_full.shape)
print("y_full:", y_full.shape)
print("X_test:", X_test.shape)

X_full: (307511, 1162)
y_full: (307511,)
X_test: (48744, 1162)


In [10]:
clean_train_cols = [
    re.sub(r"[^A-Za-z0-9_]+", "_", str(col))
    for col
    in X_full.columns
]

clean_test_cols = [
    re.sub(r"[^A-Za-z0-9_]+", "_", str(col))
    for col
    in X_test.columns
]

In [11]:
X_full.columns = (clean_train_cols)
X_test.columns = (clean_test_cols)

In [13]:
cat_cols = (X_full.select_dtypes(include="object").columns.tolist())

In [14]:
for col in cat_cols:

    X_full[col] = (X_full[col].astype("category"))
    X_test[col] = (pd.Categorical(X_test[col], categories=(X_full[col].cat.categories)))

In [18]:
numeric_kmeans_features = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "BUREAU_LAST_10_DEBT_TO_CREDIT_max",
    "INST_LAST_730D_LATE_DAYS_MEAN",
    "DAYS_BIRTH",
    "BUREAU_MAX_DEBT_RATIO",
    "DAYS_EMPLOYED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "PREV_LAST_10_TOTAL_PAYMENT_TO_CREDIT_max",
    "BUREAU_LAST_5_DEBT_TO_CREDIT_median",
    "INST_PAID_SUM",
    "INST_LATE_PAYMENT_SHARE"
]

categorical_kmeans_features = [
    "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE",
    "CODE_GENDER",
    "NAME_FAMILY_STATUS"
]

kmeans_features = (numeric_kmeans_features + categorical_kmeans_features)

print("Всего:", len(kmeans_features))

Всего: 20


In [19]:
X_kmeans_full = (X_full[kmeans_features].copy())
X_kmeans_test = (X_test[kmeans_features].copy())

print(X_kmeans_full.shape, X_kmeans_test.shape)
show_ram()

(307511, 20) (48744, 20)
RAM Python: 1.98 ГБ


In [20]:
X_kmeans_full["DAYS_EMPLOYED"] = (X_kmeans_full["DAYS_EMPLOYED"].replace(365243, np.nan))
X_kmeans_test["DAYS_EMPLOYED"] = (X_kmeans_test["DAYS_EMPLOYED"].replace(365243, np.nan))

In [21]:
clip_bounds = {}
for col in numeric_kmeans_features:
    lower = (X_kmeans_full[col].quantile(0.01))
    upper = (X_kmeans_full[col].quantile(0.99))
    clip_bounds[col] = (lower, upper)
    X_kmeans_full[col] = (X_kmeans_full[col].clip(lower=lower, upper=upper))
    X_kmeans_test[col] = (X_kmeans_test[col].clip(lower=lower, upper=upper))

In [23]:
numeric_pipeline = Pipeline([
(
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )
    )
])

kmeans_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_kmeans_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_kmeans_features
        )
    ],
    sparse_threshold=1.0
)

In [24]:
%%time

X_kmeans_full_ready = (kmeans_preprocessor.fit_transform(X_kmeans_full))
X_kmeans_test_ready = (kmeans_preprocessor.transform(X_kmeans_test))
X_kmeans_full_ready = (X_kmeans_full_ready.astype(np.float32))
X_kmeans_test_ready = (X_kmeans_test_ready.astype(np.float32))

Train KMeans: (307511, 105)
Test KMeans: (48744, 105)
<class 'scipy.sparse._csr.csr_matrix'>
RAM Python: 2.46 ГБ
CPU times: user 896 ms, sys: 81 ms, total: 977 ms
Wall time: 995 ms


In [25]:
%%time

kmeans_final = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)
kmeans_final.fit(X_kmeans_full_ready)

CPU times: user 13 s, sys: 54.9 ms, total: 13.1 s
Wall time: 2.8 s


,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",3
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",10
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary <random_state>`.",42
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'
Name,Type,Value
"cluster_centers_ cluster_centers_: ndarray of shape (n_clusters, n_features)Coordinates of cluster centers. If the algorithm stops before fullyconverging (see ``tol`` and ``max_iter``), these will not beconsistent with ``labels_``.","ndarray[float32](3, 105)","[[ 

In [26]:
train_clusters = (kmeans_final.predict(X_kmeans_full_ready))
test_clusters = (kmeans_final.predict(X_kmeans_test_ready))
train_distances = (kmeans_final.transform(X_kmeans_full_ready))
test_distances = (kmeans_final.transform(X_kmeans_test_ready))

In [28]:
cluster_categories = [0, 1, 2]
X_full["KMEANS_3_CLUSTER"] = pd.Categorical(train_clusters, categories=cluster_categories)
X_test["KMEANS_3_CLUSTER"] = pd.Categorical(test_clusters, categories=cluster_categories)

for i in range(3):

    X_full[f"KMEANS_3_DISTANCE_{i}"] = (train_distances[:, i].astype("float32"))
    X_test[f"KMEANS_3_DISTANCE_{i}"] = (test_distances[:, i].astype("float32"))

In [30]:
del X_kmeans_full
del X_kmeans_test
del X_kmeans_full_ready
del X_kmeans_test_ready
del train_clusters
del test_clusters
del train_distances
del test_distances
del kmeans_preprocessor
del kmeans_final

free_memory()
show_ram()

RAM Python: 1.85 ГБ


In [31]:
cat_cols_final = (cat_cols + ["KMEANS_3_CLUSTER"])

In [33]:
best_params_window = {
    "max_depth": 10,
    "num_leaves": 30,
    "learning_rate": 0.015220942890410365,
    "min_child_samples": 194,
    "min_split_gain": 0.7038580548468286,
    "subsample": 0.5799009823000935,
    "colsample_bytree": 0.4311082776233844
}
best_iteration_final = 1302

In [35]:
%%time

lgbm_window_k3_final = lgb.LGBMClassifier(
        objective = "binary",
        n_estimators = best_iteration_final,
        **best_params_window,
        subsample_freq = 1,
        class_weight="balanced",
        random_state = 42,
        n_jobs = -1,
        verbosity = -1
    )

lgbm_window_k3_final.fit(X_full, y_full, categorical_feature=cat_cols_final)
test_pred_proba = lgbm_window_k3_final.predict_proba(X_test)[:, 1]

CPU times: user 31min 32s, sys: 20.8 s, total: 31min 53s
Wall time: 2min 42s


In [36]:
submission = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "TARGET": test_pred_proba
})

print(submission.shape)
display(submission.head())
print(submission.isna().sum())

(48744, 2)


,SK_ID_CURR,TARGET
0,100001.0,0.295327
1,100005.0,0.622349
2,100013.0,0.309829
3,100028.0,0.224380
4,100038.0,0.685431


SK_ID_CURR    0
TARGET        0
dtype: int64


In [37]:
submission_path = os.path.join(path_to_data, "submission_windows_kmeans_k3.csv")
submission.to_csv(submission_path, index=False)